# NYC Taxi Trip Duration — from EDA to submission

**Question:** Given information known when a taxi trip starts, how many seconds will it take? Each row is one trip and `trip_duration` is the target. This is regression because the answer is a number. We inspect the data, explain feature choices, compare models fairly, and create `submission.csv` for Kaggle. The notebook uses both the competition's train and test files.


## 1. Load the data and reserve validation trips

`train.csv` has the answers; `test.csv` does not. We set aside 20% of train before using durations in EDA or model selection. The competition mixes train and test trips from the same period, so a fixed random split is useful for this competition. It does not measure performance on future months.


In [ ]:
from pathlib import Path

def locate_data_dir(kaggle_root=Path('/kaggle/input'), local_dir=Path('data')):
    slug = 'nyc-taxi-trip-duration'
    candidates = [kaggle_root / slug, kaggle_root / 'competitions' / slug, local_dir]
    for folder in candidates:
        has_train = (folder / 'train.csv').is_file() or (folder / 'train.zip').is_file()
        has_test = (folder / 'test.csv').is_file() or (folder / 'test.zip').is_file()
        if has_train and has_test:
            return folder
    raise FileNotFoundError('NYC Taxi train/test files not found. Attach the competition data in Kaggle Inputs.')


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor

data_dir = locate_data_dir()
train_file = data_dir / 'train.csv'
if not train_file.exists():
    train_file = data_dir / 'train.zip'
train = pd.read_csv(train_file, parse_dates=['pickup_datetime'])
train_idx, valid_idx = train_test_split(np.arange(len(train)), test_size=0.20, random_state=42)
fit_eda = train.iloc[train_idx].copy()
print('Full train rows, columns:', train.shape)
print('EDA/training rows:', len(fit_eda), '| validation rows:', len(valid_idx))
display(train.head())


## 2. Inspect quality and explore the data

First check types, missing values, repeated rows/IDs, and impossible values. These counts describe the raw file; the plots below use only the training portion so we do not inspect validation answers while choosing features. A duplicate row or a long trip needs investigation before deleting it; this first pass keeps the raw training rows so the comparison stays reproducible.


In [ ]:
quality_checks = pd.Series({
    'rows': len(train),
    'exact_duplicate_rows': int(train.duplicated().sum()),
    'duplicate_trip_ids': int(train['id'].duplicated().sum()),
    'nonpositive_duration': int(train['trip_duration'].le(0).sum()),
    'nonpositive_passenger_count': int(train['passenger_count'].le(0).sum()),
    'invalid_latitude': int((train[['pickup_latitude', 'dropoff_latitude']].abs() > 90).any(axis=1).sum()),
    'invalid_longitude': int((train[['pickup_longitude', 'dropoff_longitude']].abs() > 180).any(axis=1).sum()),
}, name='count')
column_check = pd.DataFrame({
    'dtype': train.dtypes.astype(str),
    'missing': train.isna().sum(),
    'unique': train.nunique(dropna=True),
})
display(column_check)
display(quality_checks.to_frame())
print('Pickup date range:', train['pickup_datetime'].min(), 'to', train['pickup_datetime'].max())


### How long do trips usually take?

The chart stops at the 99th percentile **for display only**. Long trips remain in training and evaluation. If the distribution has a long right tail, comparing errors on a log scale helps keep unusually long rides from dominating the score.


In [ ]:
import matplotlib.pyplot as plt
duration_min = fit_eda['trip_duration'] / 60
chart_limit = duration_min.quantile(0.99)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(duration_min[duration_min.between(0, chart_limit)], bins=60, color='#2874a6')
ax.set(title='Trip duration: training portion, up to 99th percentile',
       xlabel='Minutes', ylabel='Trips')
plt.tight_layout()
plt.show()
display(fit_eda['trip_duration'].describe(percentiles=[0.5, 0.9, 0.99]).to_frame('seconds'))


**What we found:** The median trip takes about **11 minutes**. The 90th percentile is about **27 minutes** and the 99th is about **57 minutes**, so duration has a long right tail. The plot hides the top 1% only to keep the common trips readable; those rows still stay in the model data.


### Does pickup time look useful?

Compare trip count and median duration by hour. They answer different questions: when rides are common, and when a typical ride takes longer. Hour is available when a new ride starts, so it is a valid model feature.


In [ ]:
import matplotlib.pyplot as plt
hourly = (fit_eda.assign(pickup_hour=fit_eda['pickup_datetime'].dt.hour)
          .groupby('pickup_hour')['trip_duration']
          .agg(trips='size', median_seconds='median')
          .reindex(range(24), fill_value=0))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(hourly.index, hourly['trips'], color='#2874a6')
axes[0].set(title='Trip count by pickup hour', xlabel='Pickup hour', ylabel='Trips')
axes[1].plot(hourly.index, hourly['median_seconds'] / 60, marker='o', color='#e67e22')
axes[1].set(title='Median duration by pickup hour', xlabel='Pickup hour', ylabel='Minutes')
for ax in axes:
    ax.set_xticks(range(0, 24, 3))
plt.tight_layout()
plt.show()


**What we found:** There are far more pickups around **18:00** than around **05:00**. Typical duration also changes with the hour: roughly **7.9 minutes at 06:00** versus **12.1 minutes at 14:00**. That supports using pickup hour as a feature, though hour alone cannot explain the whole trip.


## 3. Create features known before the trip ends

We can use pickup time, the starting and destination coordinates, passenger count, vendor, and a straight-line distance calculated from those coordinates. Straight-line distance is an approximation, not the driving route. We exclude `id` (an identifier), `dropoff_datetime` (known only after arrival), and `trip_duration` (the answer).


In [ ]:
def build_features(df):
    pickup = pd.to_datetime(df['pickup_datetime'])
    lat1 = np.radians(df['pickup_latitude'].astype(float))
    lat2 = np.radians(df['dropoff_latitude'].astype(float))
    dlat = lat2 - lat1
    dlon = np.radians(df['dropoff_longitude'].astype(float) - df['pickup_longitude'].astype(float))
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2

    features = pd.DataFrame(index=df.index)
    features['distance_km'] = 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    features['pickup_hour'] = pickup.dt.hour
    features['pickup_weekday'] = pickup.dt.dayofweek
    features['pickup_month'] = pickup.dt.month
    features['pickup_latitude'] = df['pickup_latitude']
    features['pickup_longitude'] = df['pickup_longitude']
    features['dropoff_latitude'] = df['dropoff_latitude']
    features['dropoff_longitude'] = df['dropoff_longitude']
    features['passenger_count'] = df['passenger_count']
    features['vendor_id'] = df['vendor_id']
    features['store_and_fwd'] = (df['store_and_fwd_flag'] == 'Y').astype(int)
    return features

X = build_features(train)
y = train['trip_duration'].to_numpy()
assert 'trip_duration' not in X and 'dropoff_datetime' not in X and 'id' not in X
print(X.columns.tolist())


## 4. Check the distance feature

We group straight-line distance into bands to see whether longer routes tend to take more time. This uses training rows only. The plot helps explain the feature choice; it does not prove that distance alone predicts every ride well.


In [ ]:
import matplotlib.pyplot as plt
distance_bins = [0, 1, 2, 3, 5, 8, 12, 20, 50, np.inf]
distance_table = pd.DataFrame({
    'distance_km': X.iloc[train_idx]['distance_km'],
    'duration_min': fit_eda['trip_duration'] / 60,
})
distance_table['distance_band'] = pd.cut(distance_table['distance_km'], distance_bins)
distance_summary = (distance_table.groupby('distance_band', observed=True)
                    .agg(trips=('duration_min', 'size'),
                         median_minutes=('duration_min', 'median')))
display(distance_summary)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(distance_summary.index.astype(str), distance_summary['median_minutes'], color='#2a9d8f')
ax.set(title='Median duration grows with straight-line distance',
       xlabel='Distance band (km)', ylabel='Median minutes')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()


**What we found:** Median duration rises from about **4.5 minutes** for trips under 1 km to about **20.9 minutes** for trips from 5–8 km. Distance looks useful, but the same distance can still take different amounts of time because of route and traffic.


## 5. Compare a simple baseline with three models

Every model sees the **same 150,000 training trips** and the same untouched validation rows. We predict `log1p(trip_duration)` because Kaggle scores with RMSLE, then convert predictions back to seconds. The constant baseline predicts one typical duration for every trip. Lower RMSLE is better.


In [ ]:
def rmsle(actual, predicted):
    predicted = np.clip(np.asarray(predicted), 0, None)
    return np.sqrt(np.mean((np.log1p(actual) - np.log1p(predicted)) ** 2))

sample_idx = np.random.default_rng(42).choice(train_idx, size=min(150_000, len(train_idx)), replace=False)
target = np.log1p(y[sample_idx])

models = {
    'Histogram Gradient Boosting': HistGradientBoostingRegressor(
        max_iter=120, max_leaf_nodes=31, learning_rate=0.08, random_state=42),
    'Random Forest': RandomForestRegressor(
        n_estimators=60, max_depth=20, min_samples_leaf=4,
        max_features=0.8, n_jobs=-1, random_state=42),
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=60, max_depth=20, min_samples_leaf=4,
        max_features=0.8, n_jobs=-1, random_state=42),
}

baseline_seconds = np.expm1(target.mean())
results = [{'model': 'Constant baseline',
            'train_rmsle': rmsle(y[sample_idx], np.full(len(sample_idx), baseline_seconds)),
            'valid_rmsle': rmsle(y[valid_idx], np.full(len(valid_idx), baseline_seconds))}]
for name, model in models.items():
    model.fit(X.iloc[sample_idx], target)
    train_pred = np.expm1(model.predict(X.iloc[sample_idx])).clip(min=0)
    valid_pred = np.expm1(model.predict(X.iloc[valid_idx])).clip(min=0)
    results.append({'model': name,
                    'train_rmsle': rmsle(y[sample_idx], train_pred),
                    'valid_rmsle': rmsle(y[valid_idx], valid_pred)})

scores = pd.DataFrame(results)
display(scores.round(5))
best_name = scores.loc[scores['model'] != 'Constant baseline'].nsmallest(1, 'valid_rmsle')['model'].iloc[0]
print('Best on this validation split:', best_name)


## 6. Read the result

Compare **validation RMSLE** with the constant baseline. Check the gap between train and validation: a much lower train error can signal overfitting. The table above is calculated by this run, so use its numbers rather than old scores from another environment. Choosing the best model on one split is useful here, but it does not guarantee the same ranking for future months or other cities.


## 7. Create `submission.csv`

Refit the validation winner on a 500,000-trip sample, predict every test trip, and save `id,trip_duration`. Kaggle expects one prediction per test row in the original ID order. The file appears under this notebook's Output after a successful run; submit that file to the competition to see its score.


In [ ]:
from sklearn.base import clone

test_file = data_dir / 'test.csv'
if not test_file.exists():
    test_file = data_dir / 'test.zip'
test = pd.read_csv(test_file, parse_dates=['pickup_datetime'])
X_test = build_features(test)
assert X.columns.equals(X_test.columns)

final_idx = np.random.default_rng(42).choice(
    np.arange(len(train)), size=min(500_000, len(train)), replace=False
)
final_model = clone(models[best_name])
final_model.fit(X.iloc[final_idx], np.log1p(y[final_idx]))
predicted_seconds = np.expm1(final_model.predict(X_test)).clip(min=0)
submission = pd.DataFrame({'id': test['id'], 'trip_duration': predicted_seconds})

assert len(submission) == len(test)
assert submission['id'].equals(test['id'])
assert np.isfinite(submission['trip_duration']).all()
assert submission['trip_duration'].ge(0).all()

output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('output')
output_dir.mkdir(exist_ok=True)
submission_path = output_dir / 'submission.csv'
submission.to_csv(submission_path, index=False)
print('Saved:', submission_path, 'Rows:', len(submission))
display(submission.head())
